# Spark Fundamentals (PySpark)

Runs entirely locally (`local[*]` uses all your cores as executors-in-one-JVM).
Requires `pip install pyspark` and a JVM (Java 11+).

## 1. Start a local SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("spark-fundamentals")
         .config("spark.sql.shuffle.partitions", "8")  # default 200 is silly locally
         .getOrCreate())
print(spark.version, "| default parallelism:", spark.sparkContext.defaultParallelism)

## 2. Build a synthetic dataset and run filter / groupBy / agg

In [ ]:
# 2M synthetic taxi-ish trips
trips = (spark.range(0, 2_000_000)
         .withColumn("city_id", (F.rand(seed=1) * 20).cast("int"))
         .withColumn("distance_km", F.round(F.rand(seed=2) * 30, 2))
         .withColumn("fare", F.round(F.col("distance_km") * 2.5 + F.rand(seed=3) * 5, 2)))

summary = (trips
           .filter(F.col("distance_km") > 1.0)
           .groupBy("city_id")
           .agg(F.count("*").alias("n_trips"),
                F.round(F.avg("fare"), 2).alias("avg_fare"),
                F.round(F.sum("fare"), 2).alias("revenue"))
           .orderBy(F.desc("revenue")))

summary.show(5)
print("partitions of source:", trips.rdd.getNumPartitions())

## 3. Read a query plan and spot the shuffle (look for `Exchange`)

In [ ]:
summary.explain()
# Everything below the Exchange runs before the shuffle (map side);
# everything above runs after it (reduce side). groupBy caused it.

## 4. Force a skewed join, then fix it with a broadcast join

In [ ]:
import time

# 95% of rows share one hot key -> one giant partition in a shuffle join
skewed = (spark.range(0, 1_000_000)
          .withColumn("key", F.when(F.rand(seed=4) < 0.95, 0)
                              .otherwise((F.rand(seed=5) * 1000).cast("int"))))
small = spark.range(0, 1001).withColumnRenamed("id", "key") \
             .withColumn("label", F.concat(F.lit("cat_"), F.col("key")))

t0 = time.time()
shuffle_join = skewed.join(small.hint("shuffle_hash"), "key").count()
t_shuffle = time.time() - t0

t0 = time.time()
broadcast_join = skewed.join(F.broadcast(small), "key").count()
t_broadcast = time.time() - t0

print(f"shuffle join:   {t_shuffle:.2f}s  ({shuffle_join} rows)")
print(f"broadcast join: {t_broadcast:.2f}s  ({broadcast_join} rows)")
# Broadcast avoids moving the big (skewed) side entirely - no shuffle, no straggler.

## 5. Write partitioned Parquet and see partition pruning

In [ ]:
out = "/tmp/spark_demo_trips"
trips.withColumn("city_id_part", F.col("city_id")) \
     .write.mode("overwrite").partitionBy("city_id_part").parquet(out)

pruned = spark.read.parquet(out).filter(F.col("city_id_part") == 3)
pruned.explain()  # PartitionFilters: only city_id_part=3 directories are read
print(pruned.count(), "rows read from 1 of 20 partitions")

## 6. Mini-project

Swap the synthetic data for a real multi-million-row dataset (e.g. NYC taxi parquet files),
repeat sections 2-5, and record here: runtime and plan before vs. after your optimization
(broadcast join, repartition, or caching an intermediate).

In [ ]:
spark.stop()